# Tugas 3 IF4073 Pemrosesan Citra Digital — Full Training (5 Warm-up + 20 Fine-tuning = 25 Epoch)
**Eksperimen**: `camvid11_full_25epochs`  
**Arsitektur**: `DeepLabV3 MobileNetV3-Large` (Inisialisasi baru dari pretrained weights resmi `torchvision`, **bukan** melanjutkan dari checkpoint pilot 2 epoch)  
**Konfigurasi**:
- **Stage 1 (Warm-up)**: `5` epoch (`backbone` frozen + BatchNorm `eval()`, `head_lr = 1e-3`)
- **Stage 2 (Fine-tuning)**: `20` epoch (`backbone` unfrozen, `backbone_lr = 1e-4`, `head_lr = 5e-4`)
- **Image Size**: `360 x 480`, **Batch Size**: `8`, **Num Workers**: `2`, **Pin Memory**: `True`, **Device**: `cuda`
- **Class Weighting**: `ENet` (dihitung dari 421 sampel `train`), **Optimizer**: `AdamW`, **Scheduler**: `ReduceLROnPlateau` (memantau validation `mIoU`)

## 1. Pemeriksaan GPU CUDA & Penyiapan Workspace (`/content/IF4073-Image-Segmentation`)

In [ ]:
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import zipfile
import requests
import torch
import torchvision

print('=== 1. VERIFIKASI GPU CUDA ===')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA Active : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU CUDA TIDAK TERSEDIA! Berhenti sesuai prosedur pengamanan (tidak menjalankan full training di CPU).'
    )

device_idx = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(device_idx)
gpu_props = torch.cuda.get_device_properties(device_idx)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)
total_vram_mb = gpu_props.total_memory / (1024 ** 2)
print(f'GPU Device  : {gpu_name} ({total_vram_gb:.2f} GB / {total_vram_mb:.0f} MB VRAM)')

PROJECT_DIR = Path('/content/IF4073-Image-Segmentation') if Path('/content').exists() else Path.cwd()
LOCAL_ZIP_PATH = Path('/content/camvid_project_bundle.zip')

if not (PROJECT_DIR / 'src' / 'deep_learning' / 'train.py').is_file():
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    if LOCAL_ZIP_PATH.is_file():
        print(f'Mengekstrak {LOCAL_ZIP_PATH} ke {PROJECT_DIR}...')
        dl_known_files = {
            '__init__.py', 'config.py', 'dataset.py', 'model.py',
            'metrics.py', 'training_config.py', 'train.py',
            'test_dataset.py', 'test_model.py', 'test_training.py',
            'visualize_dataset.py', 'visualize_predictions.py', 'plot_training_history.py'
        }
        with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
            for info in zf.infolist():
                norm_name = info.filename.replace('\\', '/').strip('/')
                if not norm_name or info.is_dir() or norm_name.endswith('/'):
                    continue
                if 'src/deep_learning/' in norm_name:
                    rel_path = 'src/deep_learning/' + norm_name.split('src/deep_learning/', 1)[1]
                elif 'deep_learning/' in norm_name:
                    rel_path = 'src/deep_learning/' + norm_name.split('deep_learning/', 1)[1]
                elif norm_name.startswith(('src/', 'data/', 'models/', 'results/')):
                    rel_path = norm_name
                elif Path(norm_name).name in dl_known_files:
                    rel_path = f'src/deep_learning/{Path(norm_name).name}'
                else:
                    rel_path = norm_name
                target_path = PROJECT_DIR / rel_path
                target_path.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(info, 'r') as src_f, open(target_path, 'wb') as dst_f:
                    shutil.copyfileobj(src_f, dst_f)
    else:
        raise FileNotFoundError(
            'File src/deep_learning/train.py belum ada dan /content/camvid_project_bundle.zip belum diunggah.'
        )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

for folder in ['src/deep_learning', 'data/CamVid', 'models', 'results/training']:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

camvid_dir = PROJECT_DIR / 'data' / 'CamVid'
if not (camvid_dir / 'train').is_dir() or len(list((camvid_dir / 'train').glob('*.png'))) != 421:
    print('Mengunduh dataset CamVid 701 sampel dari mirror terverifikasi...')
    tmp_repo = PROJECT_DIR / 'data' / '_tmp_camvid_repo'
    if tmp_repo.exists():
        shutil.rmtree(tmp_repo)
    subprocess.run([
        'git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
        'https://github.com/GeorgeSeif/Semantic-Segmentation-Suite.git', str(tmp_repo)
    ], check=True)
    subprocess.run(['git', '-C', str(tmp_repo), 'sparse-checkout', 'set', 'CamVid'], check=True)
    for item in (tmp_repo / 'CamVid').iterdir():
        dest = camvid_dir / item.name
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        shutil.move(str(item), str(dest))
    shutil.rmtree(tmp_repo, ignore_errors=True)
    wb_url = 'https://web.archive.org/web/20231125001718id_/http://web4.cs.ucl.ac.uk/staff/g.brostow/MotionSegRecData/data/label_colors.txt'
    try:
        r = requests.get(wb_url, timeout=30)
        if r.status_code == 200:
            (camvid_dir / 'label_colors.txt').write_bytes(r.content)
    except Exception:
        pass

for split_name, expected in [('train', 421), ('val', 112), ('test', 168)]:
    n_img = len(list((camvid_dir / split_name).glob('*.png')))
    n_lbl = len(list((camvid_dir / f'{split_name}_labels').glob('*.png')))
    assert n_img == expected and n_lbl == expected
    print(f'  [OK] Split {split_name:<5}: {n_img} citra & {n_lbl} label')

## 2. Jalankan Full Training (`camvid11_full_25epochs` — 5 Epoch Warm-up + 20 Epoch Fine-tuning)

- **Inisialisasi**: Menggunakan model baru dengan bobot pretrained resmi `torchvision` (`DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1`), **tidak** memakai checkpoint pilot.
- **Proteksi Terputus (Auto-Resume)**: Jika sesi Colab sempat terputus di tengah jalan pada eksperimen `camvid11_full_25epochs` yang sama (`epoch < 25`), sel ini akan otomatis mendeteksi `models/camvid11_full_25epochs_last.pth` dan melanjutkan dari epoch terakhir.

In [ ]:
import json
import time
import torch
from src.deep_learning.training_config import TrainingConfig
from src.deep_learning.train import run_training, load_training_checkpoint
from src.deep_learning.config import CLASS_NAMES_11

assert torch.cuda.is_available(), 'GPU CUDA wajib tersedia untuk Full Training!'
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

EXP_NAME = 'camvid11_full_25epochs'
models_dir = PROJECT_DIR / 'models'
results_dir = PROJECT_DIR / 'results' / 'training'
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

full_config = TrainingConfig(
    data_dir=str(PROJECT_DIR / 'data' / 'CamVid'),
    image_size=(360, 480),
    batch_size=8,
    num_workers=2,
    pin_memory=True,
    device='cuda',
    seed=42,
    pretrained=True,
    aux_loss=True,
    aux_loss_weight=0.4,
    warmup_epochs=5,
    warmup_head_lr=1e-3,
    finetune_epochs=20,
    finetune_backbone_lr=1e-4,
    finetune_head_lr=5e-4,
    weight_decay=1e-4,
    use_class_weights=True,
    class_weight_method='enet',
    scheduler_factor=0.5,
    scheduler_patience=3,
    min_lr=1e-6,
    grad_clip_norm=5.0,
    use_augmentation=True,
    models_dir=str(models_dir),
    results_dir=str(results_dir),
    experiment_name=EXP_NAME,
    overwrite=True,
)

# Simpan konfigurasi eksperimen ke JSON
config_json_path = results_dir / f'{EXP_NAME}_config.json'
with open(config_json_path, 'w', encoding='utf-8') as f:
    json.dump(full_config.to_dict(), f, indent=2)
print(f'Konfigurasi eksperimen disimpan ke: {config_json_path}')

# Cek apakah ada checkpoint last dari eksperimen full_25epochs yang belum selesai (proteksi disconnect)
last_full_ckpt = models_dir / f'{EXP_NAME}_last.pth'
resume_ckpt = None
if last_full_ckpt.is_file():
    _, _, _, _, last_meta = load_training_checkpoint(last_full_ckpt, device='cpu')
    done_ep = int(last_meta.get('epoch', 0))
    if 0 < done_ep < full_config.total_epochs:
        print(f'[AUTO-RESUME] Ditemukan checkpoint {last_full_ckpt.name} pada epoch {done_ep}/{full_config.total_epochs}. Melanjutkan...')
        resume_ckpt = last_full_ckpt

t0 = time.perf_counter()
full_output = run_training(full_config, resume_checkpoint=resume_ckpt)
total_wall_seconds = time.perf_counter() - t0

peak_vram_allocated_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
peak_vram_reserved_mb = torch.cuda.max_memory_reserved() / (1024 ** 2)
print(f'\nFull training 25 epoch selesai dalam {total_wall_seconds:.2f} detik ({total_wall_seconds / 60:.2f} menit).')

## 3. Evaluasi Hasil Full Training, Pembuatan Grafik, & Verifikasi Inferensi Best Checkpoint

In [ ]:
import csv
import json
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from src.deep_learning.config import CLASS_NAMES_11, NUM_CLASSES
from src.deep_learning.dataset import CamVidDataset
from src.deep_learning.model import load_checkpoint

paths = full_output['paths']
csv_path = Path(paths['csv_log'])
best_ckpt_path = Path(paths['best_checkpoint'])
last_ckpt_path = Path(paths['last_checkpoint'])
summary_json_path = Path(paths['summary_json'])

# Baca seluruh baris dari CSV history
rows = []
with open(csv_path, 'r', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        parsed = {}
        for k, v in r.items():
            if k == 'stage':
                parsed[k] = v
            elif k in ('epoch', 'stage_epoch'):
                parsed[k] = int(float(v))
            else:
                parsed[k] = float(v)
                assert math.isfinite(parsed[k]), f'Nilai non-finite pada {k}: {v}'
        rows.append(parsed)

epochs = [r['epoch'] for r in rows]
train_loss = [r['train_loss'] for r in rows]
val_loss = [r['val_loss'] for r in rows]
train_miou = [r['train_miou'] * 100.0 for r in rows]
val_miou = [r['val_miou'] * 100.0 for r in rows]
train_acc = [r['train_pixel_acc'] * 100.0 for r in rows]
val_acc = [r['val_pixel_acc'] * 100.0 for r in rows]
bb_lr = [r['backbone_lr'] for r in rows]
hd_lr = [r['head_lr'] for r in rows]

best_idx = int(np.argmax(val_miou))
best_row = rows[best_idx]
best_ep = best_row['epoch']
best_miou_pct = best_row['val_miou'] * 100.0
best_val_acc_pct = best_row['val_pixel_acc'] * 100.0

warmup_time_s = sum(r['epoch_seconds'] for r in rows if r['stage'] == 'warmup')
finetune_time_s = sum(r['epoch_seconds'] for r in rows if r['stage'] == 'finetune')
total_epoch_time_s = warmup_time_s + finetune_time_s
warmup_epochs = sum(1 for r in rows if r['stage'] == 'warmup')

# Buat 4 grafik training (Loss, mIoU/Acc, Learning Rate, dan Combined Dashboard)
loss_plot_path = results_dir / f'{EXP_NAME}_loss_curve.png'
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(epochs, train_loss, 'o-', label='Train Loss', color='#1f77b4', linewidth=2, markersize=4.5)
ax.plot(epochs, val_loss, 's-', label='Validation Loss', color='#d62728', linewidth=2, markersize=4.5)
ax.axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.3, label='Warm-up -> Fine-tune')
ax.axvline(best_ep, color='#2ca02c', linestyle=':', linewidth=1.5, label=f'Best Val mIoU (Epoch {best_ep})')
ax.set_title(f'Train vs Validation Loss ({EXP_NAME})', fontsize=12.5, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_xticks(epochs)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(loss_plot_path, dpi=140)
plt.close(fig)

miou_plot_path = results_dir / f'{EXP_NAME}_miou_curve.png'
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(epochs, val_miou, 's-', label='Validation mIoU (%)', color='#2ca02c', linewidth=2.2, markersize=4.5)
ax.plot(epochs, train_miou, 'o--', label='Train mIoU (%)', color='#7fbc41', linewidth=1.7, markersize=3.5)
ax.plot(epochs, val_acc, '^-', label='Validation Pixel Acc (%)', color='#9467bd', linewidth=2, markersize=4.5)
ax.plot(epochs, train_acc, 'v--', label='Train Pixel Acc (%)', color='#c5b0d5', linewidth=1.5, markersize=3.5)
ax.scatter([best_ep], [best_miou_pct], color='gold', edgecolor='black', s=110, zorder=6,
           label=f'Best Val mIoU: {best_miou_pct:.2f}% (Ep {best_ep})')
ax.axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.3, label='Warm-up -> Fine-tune')
ax.set_title(f'Validation mIoU & Pixel Accuracy per Epoch ({EXP_NAME})', fontsize=12.5, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Percentage (%)')
ax.set_xticks(epochs)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(miou_plot_path, dpi=140)
plt.close(fig)

lr_plot_path = results_dir / f'{EXP_NAME}_lr_curve.png'
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(epochs, hd_lr, 's-', label='Classifier Head LR', color='#ff7f0e', linewidth=2, markersize=4.5)
ax.plot(epochs, bb_lr, 'o-', label='Backbone LR', color='#17becf', linewidth=2, markersize=4.5)
ax.axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.3, label='Warm-up -> Fine-tune')
ax.set_title(f'Learning Rate Schedule ({EXP_NAME})', fontsize=12.5, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_xticks(epochs)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(lr_plot_path, dpi=140)
plt.close(fig)

combined_plot_path = results_dir / f'{EXP_NAME}_curves_combined.png'
fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))
fig.suptitle(
    f'DeepLabV3 MobileNetV3-Large CamVid 11-Class Full Training ({EXP_NAME}) — '
    f'Best Val mIoU: {best_miou_pct:.2f}% @ Epoch {best_ep}',
    fontsize=13.5, fontweight='bold', y=0.98
)
axes[0].plot(epochs, train_loss, 'o-', label='Train Loss', color='#1f77b4', linewidth=2, markersize=4)
axes[0].plot(epochs, val_loss, 's-', label='Val Loss', color='#d62728', linewidth=2, markersize=4)
axes[0].axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.2, label='Stage 1 -> 2')
axes[0].axvline(best_ep, color='#2ca02c', linestyle=':', linewidth=1.4, label=f'Best Ep {best_ep}')
axes[0].set_title('1. Train & Validation Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(True, linestyle='--', alpha=0.4); axes[0].legend(fontsize=8.5)

axes[1].plot(epochs, val_miou, 's-', label='Val mIoU (%)', color='#2ca02c', linewidth=2.2, markersize=4)
axes[1].plot(epochs, train_miou, 'o--', label='Train mIoU (%)', color='#7fbc41', linewidth=1.6, markersize=3.5)
axes[1].plot(epochs, val_acc, '^-', label='Val Pixel Acc (%)', color='#9467bd', linewidth=2, markersize=4)
axes[1].scatter([best_ep], [best_miou_pct], color='gold', edgecolor='black', s=95, zorder=6, label=f'Best: {best_miou_pct:.2f}%')
axes[1].axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.2)
axes[1].set_title('2. Validation mIoU & Pixel Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Percentage (%)'); axes[1].grid(True, linestyle='--', alpha=0.4); axes[1].legend(fontsize=8.5)

axes[2].plot(epochs, hd_lr, 's-', label='Head LR', color='#ff7f0e', linewidth=2, markersize=4)
axes[2].plot(epochs, bb_lr, 'o-', label='Backbone LR', color='#17becf', linewidth=2, markersize=4)
axes[2].axvline(warmup_epochs + 0.5, color='gray', linestyle='--', linewidth=1.2, label='Stage 1 -> 2')
axes[2].set_title('3. Learning Rate Schedule', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate'); axes[2].grid(True, linestyle='--', alpha=0.4); axes[2].legend(fontsize=8.5)

fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(combined_plot_path, dpi=140)
plt.show()
plt.close(fig)

# Muat kembali Best Checkpoint dan uji inferensi pada 1 sampel validation
loaded_best_model, loaded_best_meta = load_checkpoint(best_ckpt_path, device='cuda')
loaded_best_model.eval()
val_ds_check = CamVidDataset(PROJECT_DIR / 'data' / 'CamVid', split='val', target_size=(360, 480), imagenet_normalize=True)
sample_img_t, _ = val_ds_check[0]
with torch.no_grad():
    sample_logits = loaded_best_model(sample_img_t.unsqueeze(0).to('cuda'))
    assert tuple(sample_logits.shape) == (1, NUM_CLASSES, 360, 480)
    assert torch.isfinite(sample_logits).all()

print('====================================================================================')
print('           LAPORAN LENGKAP FULL TRAINING 25 EPOCH (camvid11_full_25epochs)          ')
print('====================================================================================')
print(f'1. Best Epoch                           : Epoch {best_ep} (Stage: {best_row["stage"]})')
print(f'2. Best Validation mIoU                 : {best_miou_pct:.2f}%')
print(f'3. Validation Pixel Acc @ Best Checkpoint: {best_val_acc_pct:.2f}%')
print(f'4. Total Waktu Training (25 Epoch)      : {total_epoch_time_s:.2f} detik ({total_epoch_time_s / 60:.2f} menit)')
print(f'   - Waktu Stage 1 (Warm-up 5 Epoch)    : {warmup_time_s:.2f} detik ({warmup_time_s / 5:.2f} dtk/epoch)')
print(f'   - Waktu Stage 2 (Fine-tune 20 Epoch) : {finetune_time_s:.2f} detik ({finetune_time_s / 20:.2f} dtk/epoch)')
print(f'   - Peak VRAM Allocated / Reserved     : {peak_vram_allocated_mb:.1f} MB / {peak_vram_reserved_mb:.1f} MB')

print('\n5. Per-Class Validation IoU (%) pada Best Checkpoint (Epoch ' + str(best_ep) + '):')
print(f"{'ID':>2} | {'Nama Kelas':<12} | {'Val IoU @ Ep 1':>15} | {'Val IoU @ Ep 5 (End Warmup)':>27} | {'Val IoU @ Best Ep ' + str(best_ep):>22}")
print('-' * 88)
ep1_row = rows[0]
ep5_row = rows[min(4, len(rows) - 1)]
for cid, cname in enumerate(CLASS_NAMES_11):
    v1 = ep1_row[f'val_iou_{cname}'] * 100.0
    v5 = ep5_row[f'val_iou_{cname}'] * 100.0
    vb = best_row[f'val_iou_{cname}'] * 100.0
    print(f'{cid:2d} | {cname:<12} | {v1:14.2f}% | {v5:26.2f}% | {vb:21.2f}%')

print('\n6. Riwayat Learning Rate & Metrik Seluruh 25 Epoch:')
print(f"{'Ep':>2} | {'Stage':<8} | {'BB LR':>8} | {'Head LR':>8} | {'Tr Loss':>8} | {'Val Loss':>8} | {'Val Acc':>8} | {'Val mIoU':>9} | {'Time':>6}")
print('-' * 86)
for r in rows:
    tag = ' *' if r['epoch'] == best_ep else ''
    print(
        f"{r['epoch']:2d} | {r['stage']:<8} | {r['backbone_lr']:8.1e} | {r['head_lr']:8.1e} | "
        f"{r['train_loss']:8.4f} | {r['val_loss']:8.4f} | {r['val_pixel_acc']*100:7.2f}% | "
        f"{r['val_miou']*100:8.2f}% | {r['epoch_seconds']:5.1f}s{tag}"
    )

print('\n7. Verifikasi Pemuatan Best Checkpoint untuk Inferensi:')
print(f'   [PASS] load_checkpoint({best_ckpt_path.name}) berhasil; forward pass (1, 3, 360, 480) -> {tuple(sample_logits.shape)} finite.')
print('====================================================================================')

## 4. Pengemasan & Unduh `camvid11_full_25epochs_artifacts.zip`

In [ ]:
import zipfile
from pathlib import Path

archive_path = PROJECT_DIR / 'camvid11_full_25epochs_artifacts.zip'
artifact_files = [
    best_ckpt_path,
    last_ckpt_path,
    csv_path,
    summary_json_path,
    config_json_path,
    loss_plot_path,
    miou_plot_path,
    lr_plot_path,
    combined_plot_path,
]

with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_files:
        assert Path(p).is_file(), f'File artefak tidak ditemukan: {p}'
        zf.write(p, arcname=Path(p).relative_to(PROJECT_DIR))
        print(f'  + {Path(p).relative_to(PROJECT_DIR)} ({Path(p).stat().st_size:,d} bytes)')

print(f'\nArsip lengkap berhasil dibuat: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.2f} MB)')
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Tidak berada di browser Colab; file ZIP tersimpan di disk lokal.')